In [1]:
!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn nltk

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
# Download dataset
!wget https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv -O spam.csv

df = pd.read_csv('spam.csv', encoding='latin-1')
df = df.iloc[:, :2]
df.columns = ['label', 'message']
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Add realistic scam examples (this fixes your bank account issue)
new_scams = [
    "Your bank account is suspended. Verify now: [fake link]",
    "Account suspended due to suspicious activity. Click here to verify immediately.",
    "Dear customer, your banking account has been locked. Login to unlock now.",
    "Urgent: Security alert! Your account is under review. Verify details now.",
    "Your bank has detected fraud. Please verify your identity to avoid suspension.",
    "Warning: Your account will be closed in 24 hours unless you verify.",
    "We have suspended your account due to unusual login attempts.",
    "Verify your bank details immediately or your account will be blocked."
]

new_df = pd.DataFrame({'message': new_scams, 'label': [1]*len(new_scams)})
df = pd.concat([df, new_df], ignore_index=True)

print("Dataset shape:", df.shape)
print(df['label'].value_counts())

--2026-06-08 06:59:16--  https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 503663 (492K) [application/octet-stream]
Saving to: ‘spam.csv’

spam.csv            100%[===================>] 491.86K  --.-KB/s    in 0.02s   

2026-06-08 06:59:16 (21.7 MB/s) - ‘spam.csv’ saved [503663/503663]

Dataset shape: (5580, 2)
label
0    4825
1     755
Name: count, dtype: int64


In [3]:
def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '[URL]', text)  # Replace links
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = text.lower().strip()
    return text

df['cleaned'] = df['message'].apply(clean_text)

In [4]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['cleaned'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42, stratify=df['label']
)

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class ScamDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = ScamDataset(train_texts, train_labels, tokenizer)
test_dataset = ScamDataset(test_texts, test_labels, tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2,
    ignore_mismatched_sizes=True   # <-- Fixes the UNEXPECTED/MISSING keys warning
).to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",           # <-- Fixed: was evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"                 # Avoid wandb logging issues
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.088690,0.974014
2,0.093132,0.051987,0.989247
3,0.093132,0.048195,0.990143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=837, training_loss=0.060636124445688486, metrics={'train_runtime': 198.3449, 'train_samples_per_second': 67.519, 'train_steps_per_second': 4.22, 'total_flos': 443500850700288.0, 'train_loss': 0.060636124445688486, 'epoch': 3.0})

In [6]:
# Evaluate
results = trainer.evaluate()
print("Test Accuracy:", results['eval_accuracy'])

def predict_scam(text):
    cleaned = clean_text(text)
    inputs = tokenizer(cleaned, truncation=True, padding=True, max_length=128, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
        confidence = probs[0][prediction].item()

    result = "🚨 **SCAM / Fraudulent**" if prediction == 1 else "✅ **Genuine**"
    print(result)
    print(f"Confidence: {confidence:.2%}")
    return result

# Test cases
predict_scam("Your bank account is suspended. Verify now: [fake link]")
predict_scam("Hey, are we still meeting for coffee tomorrow?")
predict_scam("Congratulations! You've won a $1000 prize. Click here to claim.")

Training Loss,Validation Loss,Epoch,Accuracy
0.093132,0.048195,3,0.990143


Test Accuracy: 0.9901433691756273
🚨 **SCAM / Fraudulent**
Confidence: 99.89%
✅ **Genuine**
Confidence: 99.96%
🚨 **SCAM / Fraudulent**
Confidence: 99.89%


'🚨 **SCAM / Fraudulent**'

In [7]:
model.save_pretrained("scam_detector_bert")
tokenizer.save_pretrained("scam_detector_bert")

from google.colab import files
!zip -r scam_detector_bert.zip scam_detector_bert/
files.download('scam_detector_bert.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: scam_detector_bert/ (stored 0%)
  adding: scam_detector_bert/tokenizer.json (deflated 71%)
  adding: scam_detector_bert/tokenizer_config.json (deflated 43%)
  adding: scam_detector_bert/model.safetensors (deflated 8%)
  adding: scam_detector_bert/config.json (deflated 49%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>